This notebook:

1. Reads a file of an MGRS tile
2. Identifies the overlapping OPERA RTC Burst IDs
3. Downloads the metadata for all the overlapping bursts from CMR/ASF
4. Saves this metadata as a parquet file
5. Downloads and organizes this data

The CMRrequests

# Get Metadata for Burst Time Series

In [12]:
from dist_s1_enumerator import get_burst_ids_in_mgrs_tiles, localize_rtc_s1_ts
from dist_s1_enumerator.asf import get_rtc_s1_ts_metadata_by_burst_ids, append_pass_data
from tqdm.auto import tqdm
import pandas as pd
import geopandas as gpd
import concurrent.futures
import warnings

In [2]:
with open('MGRS_tiles.txt') as f:
    mgrs_tile_ids = f.readlines()
mgrs_tile_ids = list(map(lambda x: x.strip(), mgrs_tile_ids))
mgrs_tile_ids[:3]

['33MXU', '20KRV', '20LMK']

In [3]:
burst_ids = get_burst_ids_in_mgrs_tiles(mgrs_tile_ids)
burst_ids[:3], len(burst_ids)

(['T058-123139-IW1', 'T058-123139-IW2', 'T058-123140-IW1'], 5303)

In [15]:
%%time

df_burst_ts = gpd.GeoDataFrame()

def get_table_for_burst_id(burst_id: str) -> gpd.GeoDataFrame():
    try:
        df_burst = get_rtc_s1_ts_metadata_by_burst_ids(burst_id)
    except ValueError:
        print(f'Burst id with mixed polarizations:', burst_id)
        df_burst = gpd.GeoDataFrame()
    return df_burst       

with warnings.catch_warnings():
    # Ignore warnings about empty dataframes
    warnings.simplefilter("ignore", category=UserWarning)
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        dfs = list(tqdm(executor.map(get_table_for_burst_id, burst_ids[:]), total=len(burst_ids)))

In [5]:
dfs[100].head()

,opera_id,jpl_burst_id,acq_dt,polarizations,track_number,pass_id,url_crosspol,url_copol,geometry
0,OPERA_L2_RTC-S1_T115-245678-IW2_20220104T14144...,T115-245678-IW2,2022-01-04 14:14:47+00:00,VV+VH,115,487,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-121.18266 43.83451, -122.3128 43.97..."
1,OPERA_L2_RTC-S1_T115-245678-IW2_20220128T14144...,T115-245678-IW2,2022-01-28 14:14:46+00:00,VV+VH,115,491,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-121.18188 43.8344, -122.312 43.9762..."
2,OPERA_L2_RTC-S1_T115-245678-IW2_20220209T14144...,T115-245678-IW2,2022-02-09 14:14:46+00:00,VV+VH,115,493,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-121.18143 43.83426, -122.31151 43.9..."
3,OPERA_L2_RTC-S1_T115-245678-IW2_20220221T14144...,T115-245678-IW2,2022-02-21 14:14:46+00:00,VV+VH,115,495,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-121.18263 43.83479, -122.31271 43.9..."
4,OPERA_L2_RTC-S1_T115-245678-IW2_20220305T14144...,T115-245678-IW2,2022-03-05 14:14:45+00:00,VV+VH,115,497,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-121.18285 43.83496, -122.31288 43.9..."


In [6]:
df_burst_ts = pd.concat(dfs, axis=0)
df_burst_ts.head()

/tmp/ipykernel_31378/1763168224.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_burst_ts = pd.concat(dfs, axis=0)


,opera_id,jpl_burst_id,acq_dt,acq_date_for_mgrs_pass,polarizations,track_number,pass_id,url_crosspol,url_copol,geometry
0,OPERA_L2_RTC-S1_T160-342228-IW2_20220107T16131...,T160-342228-IW2,2022-01-07 16:13:18+00:00,NaN,VV+VH,160,488,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-146.49615 61.4333, -148.19696 61.58..."
1,OPERA_L2_RTC-S1_T160-342228-IW2_20220119T16131...,T160-342228-IW2,2022-01-19 16:13:17+00:00,NaN,VV+VH,160,490,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-146.49705 61.43354, -148.19784 61.5..."
2,OPERA_L2_RTC-S1_T160-342228-IW2_20220131T16131...,T160-342228-IW2,2022-01-31 16:13:17+00:00,NaN,VV+VH,160,492,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-146.49811 61.43371, -148.19893 61.5..."
3,OPERA_L2_RTC-S1_T160-342228-IW2_20220212T16131...,T160-342228-IW2,2022-02-12 16:13:17+00:00,NaN,VV+VH,160,494,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-146.49734 61.43326, -148.19804 61.5..."
4,OPERA_L2_RTC-S1_T160-342228-IW2_20220224T16131...,T160-342228-IW2,2022-02-24 16:13:17+00:00,NaN,VV+VH,160,496,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,https://datapool.asf.alaska.edu/RTC/OPERA-S1/O...,"POLYGON ((-146.49673 61.43327, -148.1974 61.58..."


In [7]:
%%time

df_burst_ts.to_parquet('burst_ts.parquet', compression='zstd', index=None)

CPU times: user 707 ms, sys: 161 ms, total: 868 ms
Wall time: 883 ms


# Serialize Data

In [19]:
# df_burst_ts = gpd.read_parquet('burst_ts.parquet')

In [20]:
# This updates metadata to include MGRS info; also for downloading data using dist-s1-enumerator which has column validation
df_burst_ts = append_pass_data(df_burst_ts, mgrs_tile_ids)

# I am not sure why these are not correctly typed.
df_burst_ts.track_number = df_burst_ts.track_number.astype(int)
df_burst_ts.pass_id = df_burst_ts.pass_id.astype(int)


In [ ]:
%%time

df_burst_ts_loc = localize_rtc_s1_ts(df_burst_ts[:], 'out/burst_ts_data', tqdm_enabled=True, max_workers=5)
df_burst_ts_loc.head()

In [ ]:
%%time

df_burst_ts_loc.to_parquet('burst_ts_loc.parquet', compression='zstd', index=None)